# Quantum Convolutional Neural Network (QCNN)

A **quantum convolution** ("quanvolutional") layer works like a classical convolution, except the filter that slides over the image is a small parametrized quantum circuit instead of a matrix of weights: each image patch is encoded into a few qubits, entangled, passed through a trainable rotation, and read out as a single expectation value — exactly the role a classical convolution kernel plays, just computed quantum-mechanically.

This notebook builds one from scratch and trains it as the first layer of a small image classifier (MNIST, digits 0 vs 1), using PyTorch's ordinary `loss.backward()` end to end — no manual gradient tricks needed, since blueqatSDK's circuits return ordinary differentiable PyTorch tensors.

Reference: Henderson et al., ["Quanvolutional Neural Networks: Powering Image Recognition with Quantum Circuits"](https://arxiv.org/abs/1904.04767) (2019).

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK torchvision

  Cloning https://github.com/blueqat/blueqatSDK to /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-ycbxcgy_
  Running command git clone --filter=blob:none --quiet https://github.com/blueqat/blueqatSDK /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-ycbxcgy_


  Resolved https://github.com/blueqat/blueqatSDK to commit fde32d1a168d32fe2da3e0ab886c796095e4e552


  Installing build dependencies ... -

 \

 done


  Getting requirements to build wheel ... -

 done


  Preparing metadata (pyproject.toml) ... - done


## Data: MNIST, digits 0 vs 1

To keep this notebook's runtime reasonable (each quantum-convolution patch is an actual small circuit simulation, not a matrix multiply), we use a binary subset (digits 0 and 1 only) and a small number of training/test images. The full 10-digit, full-dataset case works exactly the same way — it's just slower, since every patch needs its own circuit run.

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_train = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
full_test = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_indices = [i for i, (_, y) in enumerate(full_train) if y in (0, 1)][:200]
test_indices = [i for i, (_, y) in enumerate(full_test) if y in (0, 1)][:100]

trainset = torch.utils.data.Subset(full_train, train_indices)
testset = torch.utils.data.Subset(full_test, test_indices)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=4, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False)

print(f"train images: {len(trainset)}, test images: {len(testset)}, batches/epoch: {len(trainloader)}")

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  0%|          | 32.8k/9.91M [00:00<00:47, 207kB/s]

  1%|          | 98.3k/9.91M [00:00<00:35, 279kB/s]

  2%|▏         | 197k/9.91M [00:00<00:24, 391kB/s] 

  4%|▍         | 426k/9.91M [00:00<00:13, 711kB/s]

...

 99%|█████████▉| 1.64M/1.65M [00:01<00:00, 1.19MB/s]

100%|██████████| 1.65M/1.65M [00:01<00:00, 876kB/s] 

  0%|          | 0.00/4.54k [00:00<?, ?B/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 7.13MB/s]

train images: 200, test images: 100, batches/epoch: 50


## The quantum convolution layer

For each $2\times2$ image patch (4 pixels, 4 qubits):

1. **Encode**: each pixel value $x$ becomes a qubit rotation, $R_y(\arctan x)$ then $R_z(\arctan x^2)$ — a standard way to map an unbounded classical value onto a bounded rotation angle.
2. **Entangle**: a ring of `CX` gates couples all 4 qubits together, so the readout below depends on the whole patch jointly, not just one pixel.
3. **Trainable rotation**: a general single-qubit rotation `u(theta, phi, lam)` on each qubit, with `theta`/`phi`/`lam` as learned parameters — this is the quantum analogue of a convolution kernel's learned weights.
4. **Read out**: the $Z$ expectation value of qubit 0 becomes this patch's single output value, exactly like a classical convolution kernel produces one output value per patch.

Sliding this over every non-overlapping $2\times2$ patch turns a $28\times28$ image into a $14\times14$ one-channel output — a genuine (quantum) convolution.

In [3]:
import math
from blueqat import Circuit
from blueqat.utils import Z
import torch.nn as nn


class Qconv(nn.Module):
    """A single-channel, 2x2, stride-2 quantum convolution layer."""

    def __init__(self, kernel_h=2, kernel_w=2):
        super().__init__()
        self.kernel_h = kernel_h
        self.kernel_w = kernel_w
        self.n_qubit = kernel_h * kernel_w
        self.params = nn.Parameter(torch.rand(self.n_qubit * 3) * 2 * math.pi)

    def _run_patch(self, patch):
        flat = patch.reshape(-1)
        circ = Circuit(self.n_qubit)
        for i in range(self.n_qubit):
            circ.ry(torch.arctan(flat[i]))[i]
            circ.rz(torch.arctan(flat[i] ** 2))[i]
        for i in range(self.n_qubit):
            circ.cx[i, (i + 1) % self.n_qubit]
        for i in range(self.n_qubit):
            circ.u(self.params[3 * i], self.params[3 * i + 1], self.params[3 * i + 2])[i]
        return circ.run(hamiltonian=Z[0])

    def forward(self, x):
        # x: (batch, 1, H, W)
        b, _, h, w = x.shape
        out_h, out_w = h // self.kernel_h, w // self.kernel_w
        out = torch.zeros(b, 1, out_h, out_w)
        for bi in range(b):
            for i in range(out_h):
                for j in range(out_w):
                    patch = x[bi, 0,
                              i * self.kernel_h:(i + 1) * self.kernel_h,
                              j * self.kernel_w:(j + 1) * self.kernel_w]
                    out[bi, 0, i, j] = self._run_patch(patch)
        return out

A quick sanity check: run one patch through the layer and confirm gradients flow back to its parameters with an ordinary `.backward()` call — no custom `autograd.Function`, no finite-difference gradient estimation. This is a direct consequence of blueqatSDK's circuits returning differentiable PyTorch tensors throughout (as used in earlier tutorials like [100_quantum_classical_hybrid.ipynb](100_quantum_classical_hybrid.ipynb)); earlier quantum-convolution implementations (including this same idea, run on the previous blueqat SDK) needed a hand-written finite-difference backward pass instead, since circuit outputs weren't differentiable tensors yet.

In [4]:
qconv = Qconv()
sample_patch = torch.rand(2, 2)
output = qconv._run_patch(sample_patch)
output.backward()
print("output:", output.item())
print("gradient reached the circuit's own parameters:", qconv.params.grad is not None)

output: -0.02169285872482843
gradient reached the circuit's own parameters: True


## The full model

`Qconv` plugs into an ordinary PyTorch model exactly like `nn.Conv2d` would: $28\times28 \to$ (quantum conv) $\to 14\times14 \to$ (max pool) $\to 7\times7 \to$ (flatten + linear) $\to$ 2 class scores.

In [5]:
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.qconv = Qconv(2, 2)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(7 * 7, 2)

    def forward(self, x):
        x = self.qconv(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


net = Net()
print(net)

Net(
  (qconv): Qconv()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=49, out_features=2, bias=True)
)


## Training

A short training loop — standard PyTorch, nothing quantum-specific about it. Each `loss.backward()` call transparently differentiates through every quantum circuit run during the forward pass.

In [6]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.1)

t0 = time.time()
for epoch in range(2):
    running_loss = 0.0
    for inputs, labels in trainloader:
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"epoch {epoch + 1}: avg loss = {running_loss / len(trainloader):.4f}")

print(f"training time: {time.time() - t0:.1f}s")

epoch 1: avg loss = 0.7665


epoch 2: avg loss = 0.2565
training time: 114.2s


In [7]:
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        outputs = net(images)
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"test accuracy: {100 * correct / total:.1f}% ({correct}/{total})")

test accuracy: 98.0% (98/100)


With only 200 training images and 2 epochs, this small quantum-convolutional model should already separate 0s from 1s well above chance (this is a comparatively easy binary task). The point of this notebook isn't to compete with a classical CNN on accuracy — with a couple hundred images and a single 4-qubit patch filter, it won't — it's to show a genuine quantum-classical hybrid convolution layer trained end to end with ordinary backpropagation, exactly the way you'd plug in any other `nn.Module`.

From here, the natural next steps mirror the original technique this notebook is based on: more output channels (multiple independent `Qconv` filters, like a classical `Conv2d(in_channels, out_channels, ...)`), stacking a second quantum-convolutional layer, or scaling up to the full 10-digit classification task (the code above needs no changes for either — only the dataset/model size).